# 15 — Cargador: conjuntos, pruebas y velocidad

El cargador (`src/cargador.py`) es la pieza entre los datos guardados y la red: une el índice de `11-dataset_training`, los atributos de negativos de `14-negativos_dificiles` y el caché de `13-cache_imagenes`, y entrega lotes `x` de forma (B, 5·L, 32, 32) en float32, normalizados con el z-score de entrenamiento, junto con la etiqueta `y`.

**Decisiones que implementa**

- **Conjuntos.** Train con los pares que tienen la historia del modelo (L_max ≥ L); val y test con el conjunto común L_max = 4, para comparar los tres modelos sobre los mismos pares. Nueva Generación no entra en ningún conjunto.
- **Muestreo por época (solo train).** Todos los positivos y, por cada uno, 2 negativos de la bolsa difícil y 2 del resto (1 : 2 : 2), sin reemplazo, con semilla = 42 + época. Bolsa difícil por defecto: `red_activa ∪ casi_positivo`; variante para S7: más `nube_fria` (p10 de C13 ≤ 228.2 K). Val y test se recorren completos.
- **Orden de canales.** Historia de la imagen más antigua a la de t; dentro de cada instante, las cinco bandas (C08, C09, C10, C13, C14): canal = j·5 + banda.

**Pruebas:** formas y tipos; igualdad con `Cache.parche` (que en 13 se verificó bit a bit contra `extraer_parche` de 10); que ningún período comparta imágenes ni ventanas de etiqueta con otro; ausencia de Nueva Generación; composición y reproducibilidad del muestreo. Al final se mide la velocidad.

In [1]:
import sys, pathlib
_raiz = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / ".gitignore").exists())
sys.path.insert(0, str(_raiz))
import entorno

In [2]:
# Configuración --------------------------------------------------------------------
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

RUTA_BASE  = Path(r"C:\Users\diani\Documents\repositories\nowcasting-bogota")
RUTA_DS    = RUTA_BASE / "datasets"
RUTA_CACHE = RUTA_BASE / "data" / "cache"
EN_MEMORIA = True          # carga el caché completo (3.14 GB) en RAM; False = lectura desde disco (memmap)

sys.path.append(str(RUTA_BASE / "src"))
from cache_imagenes import Cache
from cargador import (NG, L_COMUN, CRITERIOS_DEFECTO, preparar_pares, DatasetPares,
                      MuestreadorEpoca, cargador, cod10)

SEED  = 42
ORDEN = ["train", "val", "test"]
TZ    = "America/Bogota"

zs        = pd.read_csv(RUTA_BASE / "data_ingestion" / "satellite" / "normalizacion_zscore.csv")
t0 = time.time()
cache     = Cache(RUTA_CACHE, zs, en_memoria=EN_MEMORIA)
print(f"Caché: {cache.img.shape} {cache.img.dtype} | en memoria: {EN_MEMORIA} | {time.time() - t0:.0f} s")
indice    = pd.read_parquet(RUTA_DS / "indice_completo.parquet")              # 11
atributos = pd.read_parquet(RUTA_DS / "negativos_atributos.parquet")          # 14
print(f"Índice: {len(indice):,} pares | atributos: {len(atributos):,} | torch {torch.__version__}, "
      f"CUDA: {torch.cuda.is_available()}")

Caché: (104745, 5, 60, 50) uint16 | en memoria: True | 3 s
Índice: 4,029,925 pares | atributos: 3,973,266 | torch 2.14.1+cu126, CUDA: True


## 1. Conjuntos

Se construyen los pares de cada combinación de período y historia que se usará en el modelado: train para L = 2, 3 y 4, y val y test con L = 4. La tabla muestra el tamaño de cada bolsa. Las cifras de train con L = 2 deben coincidir con las de `14-negativos_dificiles` (96,333 difíciles).

In [3]:
conjuntos, filas = {}, []
for p in ORDEN:
    for L in ((2, 3, 4) if p == "train" else (L_COMUN,)):
        d = preparar_pares(indice, atributos, cache, periodo=p, L=L)
        conjuntos[(p, L)] = d
        n = d["bolsa"].value_counts()
        filas.append({"periodo": p, "L": L, "pares": len(d), "positivos": n.get("positivo", 0),
                      "difíciles": n.get("dificil", 0), "aleatorios": n.get("aleatorio", 0),
                      "difíciles por positivo": round(n.get("dificil", 0) / n.get("positivo", 1), 1),
                      "estaciones": d["codigoestacion"].nunique()})
tabla_conj = pd.DataFrame(filas)
print(tabla_conj.to_string(index=False))

assert all(NG not in set(d["codigoestacion"]) for d in conjuntos.values()), "Nueva Generación presente"
assert tabla_conj.query("periodo == 'train' and L == 2")["difíciles"].item() == 96_333
print("\nNueva Generación ausente en todos los conjuntos; bolsa difícil de train (L = 2) igual a 14")

periodo  L   pares  positivos  difíciles  aleatorios  difíciles por positivo  estaciones
  train  2 1244157       4249      96333     1143575                    22.7          61
  train  3 1241422       4202      95709     1141511                    22.8          61
  train  4 1238827       4155      95242     1139430                    22.9          61
    val  4  325425       1487      50593      273345                    34.0          62
   test  4 2395490      12320     358865     2024305                    29.1          62

Nueva Generación ausente en todos los conjuntos; bolsa difícil de train (L = 2) igual a 14


## 2. Pruebas

### 2.1 Formas, tipos y etiquetas

In [4]:
for L in (2, 3, 4):
    d  = conjuntos[("train", L)]
    ds = DatasetPares(d, cache, L)
    mu = MuestreadorEpoca(d, 2, 2, semilla=SEED)
    x, y = next(iter(cargador(ds, mu, tam_lote=64)))
    assert x.shape == (64, 5 * L, 32, 32) and x.dtype == torch.float32, x.shape
    assert y.shape == (64,) and set(y.unique().tolist()) <= {0.0, 1.0}
    assert torch.isfinite(x).all()
    print(f"L = {L}: x {tuple(x.shape)} {x.dtype} | media {x.mean():+.2f}, desvío {x.std():.2f} | positivos en el lote: {int(y.sum())}")

L = 2: x (64, 10, 32, 32) torch.float32 | media -0.30, desvío 0.97 | positivos en el lote: 18
L = 3: x (64, 15, 32, 32) torch.float32 | media -0.23, desvío 0.96 | positivos en el lote: 9
L = 4: x (64, 20, 32, 32) torch.float32 | media -0.19, desvío 0.99 | positivos en el lote: 16


### 2.2 Igualdad con `Cache.parche`

Para una muestra de pares de cada período, cada bloque de cinco canales del lote debe coincidir con el parche que devuelve `Cache.parche` en el instante correspondiente de la historia. `Cache.parche` trabaja en float64 y el cargador en float32, por eso la tolerancia es del orden del redondeo de float32 (no una diferencia de datos).

In [5]:
rng = np.random.default_rng(SEED)
peor = 0.0
for (p, L), d in conjuntos.items():
    ds = DatasetPares(d, cache, L)
    idx = rng.choice(len(d), 300, replace=False)
    x, y = ds[idx]
    x = x.numpy()
    assert (y.numpy() == d["etiqueta"].to_numpy(np.float32)[idx]).all()
    for k, i in enumerate(idx):
        r = d.iloc[i]
        for j in range(L):
            t_j = (r["timestamp"] - (L - 1 - j) * pd.Timedelta(minutes=10)).tz_convert(None)
            ref = cache.parche(t_j, int(r["codigoestacion"]))
            peor = max(peor, float(np.abs(x[k, j * 5:(j + 1) * 5] - ref).max()))
print(f"Diferencia máxima con Cache.parche (z-score): {peor:.2e}")
assert peor < 1e-4

Diferencia máxima con Cache.parche (z-score): 2.80e-06


### 2.3 Sin cruce entre períodos

Dos condiciones: (1) ninguna imagen del caché la usan pares de dos períodos distintos, y (2) la ventana de la etiqueta del último par de un período (t + 3 h) termina antes del primer par del período siguiente, que es lo que garantiza la purga de 3 h de 11. Se comprueban con L = 4, la historia más larga.

La historia de los primeros pares de un período (hasta 30 min antes del corte) coincide en el tiempo con los últimos minutos de lluvia de las ventanas de etiqueta del período anterior. No es una fuga: son imágenes del período siguiente frente a registros de pluviómetro del anterior, y ninguna etiqueta de un período queda al alcance de las entradas de otro. Se reporta el margen para dejarlo explícito.

In [6]:
usadas = {p: set(np.unique(conjuntos[(p, 4)][[f"fila_{j}" for j in range(4)]].to_numpy())) for p in ORDEN}
for a, b in [("train", "val"), ("val", "test"), ("train", "test")]:
    comun = usadas[a] & usadas[b]
    assert not comun, f"{a} y {b} comparten {len(comun)} imágenes"

for a, b in [("train", "val"), ("val", "test")]:
    fin_etiqueta = conjuntos[(a, 4)]["timestamp"].max() + pd.Timedelta(hours=3)
    primer_t     = conjuntos[(b, 4)]["timestamp"].min()
    inicio_hist  = primer_t - 3 * pd.Timedelta(minutes=10)
    print(f"{a} → {b}: última ventana de etiqueta de {a} termina {fin_etiqueta.tz_convert(TZ):%Y-%m-%d %H:%M}; "
          f"primer par de {b} en {primer_t.tz_convert(TZ):%Y-%m-%d %H:%M} "
          f"(su historia empieza {inicio_hist.tz_convert(TZ):%H:%M}), hora local")
    assert fin_etiqueta < primer_t
print("Sin imágenes compartidas entre períodos; ninguna ventana de etiqueta alcanza el período siguiente")

train → val: última ventana de etiqueta de train termina 2024-11-14 22:50; primer par de val en 2024-11-15 00:00 (su historia empieza 23:30), hora local
val → test: última ventana de etiqueta de val termina 2024-12-31 23:50; primer par de test en 2025-01-01 00:00 (su historia empieza 23:30), hora local
Sin imágenes compartidas entre períodos; ninguna ventana de etiqueta alcanza el período siguiente


### 2.4 Muestreo por época

En cada época deben estar todos los positivos, exactamente 2 difíciles y 2 aleatorios por positivo, sin repetidos. La misma época con la misma semilla debe dar la misma muestra; épocas distintas, negativos distintos.

In [7]:
d  = conjuntos[("train", 2)]
mu = MuestreadorEpoca(d, n_dificiles=2, n_aleatorios=2, semilla=SEED)
P  = int((d["bolsa"] == "positivo").sum())

muestras = {}
for e in (0, 1):
    mu.fijar_epoca(e)
    muestras[e] = mu.indices()
    c = d["bolsa"].iloc[muestras[e]].value_counts()
    assert c["positivo"] == P and c["dificil"] == 2 * P and c["aleatorio"] == 2 * P
    assert len(np.unique(muestras[e])) == len(muestras[e])
    print(f"Época {e}: {len(muestras[e]):,} pares → {c.to_dict()}")

mu.fijar_epoca(0)
assert (mu.indices() == muestras[0]).all(), "La época 0 no es reproducible"
neg = lambda m: set(m[d["bolsa"].iloc[m].to_numpy() != "positivo"])
compartidos = len(neg(muestras[0]) & neg(muestras[1])) / len(neg(muestras[0]))
print(f"Negativos que se repiten entre la época 0 y la 1: {compartidos:.1%}")
assert compartidos < 0.5

# Variante con nube_fria: solo cambia la bolsa difícil
d_nf = preparar_pares(indice, atributos, cache, periodo="train", L=2,
                      criterios=CRITERIOS_DEFECTO + ("nube_fria",))
print(f"Bolsa difícil con nube_fria (train, L = 2): {int((d_nf['bolsa'] == 'dificil').sum()):,}")

Época 0: 21,245 pares → {'aleatorio': 8498, 'dificil': 8498, 'positivo': 4249}
Época 1: 21,245 pares → {'aleatorio': 8498, 'dificil': 8498, 'positivo': 4249}
Negativos que se repiten entre la época 0 y la 1: 4.7%
Bolsa difícil con nube_fria (train, L = 2): 163,993


## 3. Velocidad

Se mide el tiempo de una época completa de train (L = 4, la historia más larga) y de una pasada completa por validación, solo leyendo datos (sin modelo). La referencia es la GPU: si el cargador entrega lotes más rápido de lo que la red los procesa, no es el cuello de botella.

In [8]:
TAM_LOTE = 256

def medir(dl, n_max=None):
    t0, n = time.time(), 0
    for k, (x, y) in enumerate(dl):
        n += len(y)
        if n_max and n >= n_max:
            break
    return n, time.time() - t0

d  = conjuntos[("train", 4)]
mu = MuestreadorEpoca(d, 2, 2, semilla=SEED)
n, s = medir(cargador(DatasetPares(d, cache, 4), mu, TAM_LOTE))
print(f"Época de train (L = 4): {n:,} pares en {s:.1f} s → {n / s:,.0f} pares/s")

d = conjuntos[("val", 4)]
n, s = medir(cargador(DatasetPares(d, cache, 4), None, TAM_LOTE))
print(f"Validación completa (L = 4): {n:,} pares en {s:.1f} s → {n / s:,.0f} pares/s")

Época de train (L = 4): 20,775 pares en 7.0 s → 2,980 pares/s
Validación completa (L = 4): 325,425 pares en 69.5 s → 4,682 pares/s


## Conclusión

El cargador construye los conjuntos del modelado sin Nueva Generación: train con 1,244,157, 1,241,422 y 1,238,827 pares para L = 2, 3 y 4 (4,249, 4,202 y 4,155 positivos, 61 estaciones), validación con 325,425 pares (1,487 positivos) y prueba con 2,395,490 (12,320 positivos), ambas sobre el conjunto común L = 4. La bolsa difícil de train coincide con la de `14-negativos_dificiles` (96,333 pares con L = 2). Los lotes tienen la forma y el tipo esperados y coinciden con `Cache.parche` hasta el redondeo de float32 (diferencia máxima 2.8·10⁻⁶ en z-score). Ningún período comparte imágenes con otro y ninguna ventana de etiqueta alcanza el período siguiente. Cada época de entrenamiento contiene todos los positivos y exactamente dos negativos difíciles y dos aleatorios por positivo (21,245 pares); el muestreo es reproducible con la semilla y entre dos épocas solo se repite el 4.7 % de los negativos. En el equipo local, una época de entrenamiento se lee en unos 7 s y la validación completa en unos 70 s.